In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.express as px

In [ ]:
karaoke_data = []
page_num = 1

for _ in range(13):
    url = f"https://eatout.ru/msk/guide/cat/karaoke-bary/?page={page_num}"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    karaoke_clubs = soup.find_all("div", class_ = "result")

    for club in karaoke_clubs:

        title_tag = club.find("a", class_="result__title")
        title = title_tag.get_text(strip = True)

        reviews_tag = club.find("div", class_="result__rating__reviews")
        if reviews_tag:
            reviews_text = reviews_tag.get_text(strip = True)
        else:
            reviews_text = "none"

        if reviews_text.split()[0].isdigit():
            reviews = int(reviews_text.split()[0])
        else:
            reviews = 0

        price_tag = club.find("div", class_="result__info__item", string = lambda text: text and "₽" in text)
        if price_tag:
            price_text = price_tag.get_text(strip=True).replace("средний чек", "")
        else:
            price_text = "none"

        if price_text.split()[0].isdigit():
            price = int(price_text.split()[0])
        else:
            price = 0

        rating_div = club.find("div", class_="rating__value")
        if rating_div and "style" in rating_div.attrs:
            stars_rating = rating_div["style"].split(":")[1].replace("%", "").replace(";", "").strip()
            rating = float(stars_rating) / 20
        else:
            rating = 0

        karaoke_data.append({"Название": title, "Отзывы": reviews, "Средний чек": price, "Рейтинг": rating})

    page_num += 1

df = pd.DataFrame(karaoke_data)
print(df)

                              Название  Отзывы  Средний чек  Рейтинг
0                  Кафе-караоке Lubeer      13            0     4.52
1       Ночной клуб и караоке Republic       3         1250     3.00
2        Ресторан-караоке Ambassadoria       7         1000     4.00
3                  Кафе-караоке Lusong      13          750     3.90
4    Бильярдный клуб 12 Футов Под Кием       0            0     5.00
..                                 ...     ...          ...      ...
135                   Ночной клуб СОВА       3         1500     3.34
136                       Рестобар ‘39       0            0     2.50
137                      Соло Одинцово       1         2500     0.82
138                         Wild Apple       0         1000     0.00
139                     Звезда караоке       0         2000     0.00

[140 rows x 4 columns]


In [ ]:
df_price_rating = df[["Название","Средний чек", "Рейтинг"]].copy()
df_price_rating.replace(0, None, inplace=True)
df_price_rating.dropna(subset=["Средний чек", "Рейтинг"], inplace = True)
price_rating = px.scatter(df_price_rating, x="Средний чек", y="Рейтинг", title="Корреляция между средним чеком и рейтингом")
price_rating.show()